In [ ]:
# each component is called a Runnable for Langchain. A chain can bind multiple runnables.
# it invokes a standard interface which executes 4 basic functions .invoke(), .batch(), .stream(), .ainvoke()

# the bost chain consists of prompt, llm chat model, output parser.

# chain = prompt | llm | parser
# all these (prompt, llm and parser) are Runnables and chain is a Runnable Sequence.

# Single Invoke
# chain.invoke()

# Batch Invoke
# you shouldn't use chain.batch() becuase there are not multiple prompts to be processed at parallel.
# if you still want to chain.batch() then,....
# chain.batch(
#   {your variable injection in the prompt}
# )

# Stream invoke
# for chunk in chain.stream({}):
#     print(chunk)

# Asynchronous Invoke
# import asyncio
# response = asyncio.run(chain.ainvoke({}))'



In [ ]:
# Runnable Lambda.

def preprocess_input(input: dict)-> dict:
    input["topic"] = input["topic"].lower().strip()
    input["topic"] = input["topic"].replace("_", " ")
    return input


def add_metadata(response: str)-> dict:
    return {
        "text": response,
        "length": str(response)
    }
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're a helpful assistant."),
        ("human", "Explain me about the {topic}")
    ]
)


llm = ChatGoogleGenerativeAI(model = "gemini-flash-latest", temperature = 0.8)
parser = StrOutputParser()

from langchain_core.runnables import RunnableLambda

preprocess = RunnableLambda(preprocess_input)
postprocess = RunnableLambda(add_metadata)

chain = preprocess | prompt | llm | parser | postprocess # here prerocess and postprocess are both RunnableLambdas.



<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.runnables.base.RunnableLambda'>


In [ ]:
print(type(chain))

print("Step -> ", chain.steps[-1])
print(type(chain.steps[-1]))

print("Step -> ", chain.steps[0])
print(type(chain.steps[0]))

print("Step -> ", chain.steps[1]) # this will show class ChatPromptTemplate but internally ChatPromptTemplate inherits the interface Runnables
print(type(chain.steps[1]))

print("Step -> ", chain.steps[2]) # same for the step 2 which is LLM. 
print(type(chain.steps[2])) # Shows class ChatGoogleGenerativeAI.

<class 'langchain_core.runnables.base.RunnableSequence'>
Step ->  RunnableLambda(add_metadata)
<class 'langchain_core.runnables.base.RunnableLambda'>
Step ->  RunnableLambda(preprocess_input)
<class 'langchain_core.runnables.base.RunnableLambda'>
Step ->  input_variables=['topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You're a helpful assistant."), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain me about the {topic}'), additional_kwargs={})]
<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
Step ->  output_version=None profile={'name': 'Gemini Flash Latest', 'release_date': '2025-09-25', 'last_updated': '2025-09-25', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_input

In [ ]:
# Runnable Parallel where you can execute multiple prompts parallely.

from langchain_core.runnables import RunnableParallel

# Two different prompts for the same input
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Give a one-line summary."),
    ("human", "{topic}")
])

detail_prompt = ChatPromptTemplate.from_messages([
    ("system", "Give a detailed technical explanation."),
    ("human", "{topic}")
])

eli5_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain like I'm 5 years old."),
    ("human", "{topic}")
])

# Run all three chains in parallel on same input
parallel_chain = RunnableParallel(
    summary=(summary_prompt | llm | parser),
    detailed=(detail_prompt | llm | parser),
    simple=(eli5_prompt | llm | parser)
)

results = parallel_chain.invoke({"topic": "neural networks"})

print(results["summary"])   # one line
print(results["detailed"])  # technical explanation
print(results["simple"])    # ELI5 version

In [ ]:
# the key difference between .batch() parallel execution and RunnableParallel execution is:
# .batch() executes one chain parallely multiple times on different inputs
# RunnableParallel executes multiple chain parallely on the same input.


In [ ]:
# Conditional Chain Routing.
from langchain_core.runnables import RunnableBranch

chain = RunnableBranch(
    (
        lambda x: len(x["input"] >= 500),
        ChatPromptTemplate(
            [  
                ("system" ,"Summarize the long text into 3 bullet points."),
                ("human", "{input}")
            ]
        ) | llm | parser
    )
    ,
    (
        lambda x: len(x["input"] <= 500),
        ChatPromptTemplate(
            [  
                ("system" ,"Extend the short text adding more content to the input."),
                ("human", "{input}")
            ]
        ) | llm | parser
    ),
    # default chain for fallback reason
    parser
)